# Gold Layer - Customer Purchase Frequency View

## Purpose
Analyze the distribution of days between customer orders to understand purchase frequency patterns and identify customer segments.

## Type
**SQL View** (lightweight, real-time)

## Input
* **Source:** `big_data.silver.orders` (3.3M rows)

## Output
* **Target:** `big_data.gold.vw_customer_purchase_frequency`
* **Rows:** 4 (frequency buckets)
* **Refresh:** Real-time (always reflects current Silver data)

## Use Cases
* 📅 Customer segmentation by purchase frequency
* 📧 Marketing campaign timing
* 🎯 Retention strategy optimization

## Why View (not Table)?
* ✅ Result is very small (4 rows)
* ✅ Query is fast (simple groupBy with CASE)
* ✅ Always synchronized with Silver
* ✅ Zero storage overhead

## SQL Logic
1. Filter orders with non-null days_since_prior_order
2. Create frequency buckets (1-7, 8-14, 15-30, 30+ days)
3. COUNT orders per bucket

## Execution
Run all cells sequentially. Expected runtime: ~15 seconds.

In [0]:
%sql
-- Customer Purchase Frequency View

CREATE OR REPLACE VIEW big_data.gold.vw_customer_purchase_frequency AS
SELECT 
  CASE
    WHEN days_since_prior_order <= 7 THEN '1-7 days'
    WHEN days_since_prior_order <= 14 THEN '8-14 days'
    WHEN days_since_prior_order <= 30 THEN '15-30 days'
    ELSE '30+ days'
  END AS days_range,
  COUNT(*) AS order_count
FROM big_data.silver.orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY days_range
ORDER BY days_range;

In [0]:
%sql
-- Verify view exists and preview frequency buckets
SELECT * FROM big_data.gold.vw_customer_purchase_frequency;